In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


##  **Introduction to Hugging Face transformers and datasets**

#### ***1. Load train.csv using the Hugging Face datasets library (do not use pandas). Use the .map() function to create a new column called combined_text that concatenates the prompt and A columns with a space in between. E.g., prompt_text A_text. What is the exact character length (total number of string characters using Python's len() function, NOT the number of tokens) of the combined_text string for the row at index 51? Note: We follow zero-indexing here.***

In [2]:
from datasets import load_dataset

In [3]:
dataset = load_dataset("csv", data_files="/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")["train"]

Generating train split: 0 examples [00:00, ? examples/s]

In [4]:
dataset 

Dataset({
    features: ['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer'],
    num_rows: 2000
})

In [5]:
dataset = dataset.map(lambda x : { "prompt_text A_text" : x["prompt"] + " " + x["A"]})

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [6]:
length =  len(dataset[51]["prompt_text A_text"])
print(f'The length of prompt_text A_text at the index[51] is : {length}')

The length of prompt_text A_text at the index[51] is : 614


#### ***2. Initialize the bert-base-uncased tokenizer. Look at the tokenizer's configuration properties: what is the exact total vocabulary size (the maximum number of unique subword tokens the model knows) hardcoded into this tokenizer?***

In [7]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

print(f'Vocabulary size: {tokenizer.vocab_size}')

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Vocabulary size: 30522


#### ***3. Transformers rely on special tokens to understand sentence boundaries. Using the bert-base-uncased tokenizer from the previous step, extract the exact integer ID assigned to the [SEP] (Separator) token.***

In [8]:
print(tokenizer.sep_token) 

[SEP]


In [9]:
print(f'[SEP] is coded with the id : {tokenizer.sep_token_id}') 

[SEP] is coded with the id : 102


#### ***4. Using the bert-base-uncased tokenizer, tokenize the entire prompt column of the train dataset simultaneously. Set padding='max_length', truncation=True, max_length=128, and return_tensors='pt' (PyTorch tensors). What is the exact geometric shape (dimensions) of the resulting input_ids tensor?***

In [16]:
prompts = list(dataset["prompt"])
encoded = tokenizer(
    prompts,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)


In [20]:
print(f'Shape of the ids: {(encoded['input_ids']).shape}')

Shape of the ids: torch.Size([2000, 128])


## **BERT/RoBERTa Architecture & Attention Mechanisms**

####  ***5. A standard bert-base-uncased model has a hidden embedding size of 768 dimensions and uses exactly 12 attention heads in each layer. In Transformer architecture, the hidden size is divided equally among the attention heads. What is the exact dimensionality (size) of each individual attention head?***

In [24]:
from transformers import AutoConfig

config = AutoConfig.from_pretrained("bert-base-uncased")

print(f'No of hidden sizes : {config.hidden_size}')          
print(f'No of attention heads : {config.num_attention_heads}')  

head_dim = config.hidden_size // config.num_attention_heads
print(f'dimesionality of each attention head : {head_dim}')                    

No of hidden sizes : 768
No of attention heads : 12
dimesionality of each attention head : 64


#### ***6. Load the bert-base-uncased model using AutoModel.from_pretrained(). Tokenize the prompt from row ID 0 using the tokenizer's default settings (do not apply any manual padding or truncation). Pass this tokenized input through the model. Look at the output object. What is the exact shape of the last_hidden_state tensor returned? Note: We follow zero-indexing here.***

In [25]:
from transformers import  AutoModel

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModel.from_pretrained("bert-base-uncased")
inputs = tokenizer(dataset[0]["prompt"], return_tensors="pt")
outputs = model(**inputs)
print(outputs.last_hidden_state.shape)

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


torch.Size([1, 31, 768])


#### ***7. Using the last_hidden_state tensor from the previous question, extract the embedding vector representing the [CLS] token (which is always the token at index 0). What is the sum of the first 5 float values in this [CLS] vector? (Round your answer to 4 decimal places).***

In [28]:
cls_embedding = outputs.last_hidden_state[0, 0]

answer = cls_embedding[:5].sum().item()

print(round(answer, 4))


-1.2001


#### ***8. Load bert-base-uncased with the parameter output_attentions=True. Tokenize the exact string "Light-ion fusion is a technique." (ensuring you set return_tensors='pt') and pass it through the model. Extract the attention matrix for the last layer (index -1) and the first attention head (head index 0). What is the exact attention weight (a float value) that the [CLS] token (token index 0) pays to the word fusion (you will need to find the specific token index for fusion in the input_ids)? (Round your answer to 4 decimal places).***

In [29]:
from transformers import AutoTokenizer, AutoModel

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModel.from_pretrained(
    "bert-base-uncased",
    output_attentions=True
)

text = "Light-ion fusion is a technique."

# Tokenize
inputs = tokenizer(text, return_tensors="pt")

# Forward pass
outputs = model(**inputs)

# See the tokenization
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
print(tokens)

# Find the index of "fusion"
fusion_idx = tokens.index("fusion")
print("Fusion index:", fusion_idx)

# Last layer, first head
attention = outputs.attentions[-1][0, 0]
print(attention.shape)   # (sequence_length, sequence_length)

# Attention from [CLS] (row 0) to "fusion" (column fusion_idx)
weight = attention[0, fusion_idx].item()

print("Attention weight:", round(weight, 4))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


['[CLS]', 'light', '-', 'ion', 'fusion', 'is', 'a', 'technique', '.', '[SEP]']
Fusion index: 4
torch.Size([10, 10])
Attention weight: 0.1025


## ***Context-Aware Embeddings***

#### ***9. Initialize the sentence-transformers/all-MiniLM-L6-v2 model. Use the model's .encode() method to generate embeddings for both the prompt and Option B for row ID 0. Calculate the cosine similarity between these two vectors specifically using the sentence_transformers.util.cos_sim() function. What is the resulting similarity score rounded to 4 decimal places? Note: We follow zero-indexing here.***

In [35]:
from datasets import load_dataset
from sentence_transformers import SentenceTransformer, util

dataset = load_dataset("csv", data_files="/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")["train"]
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")


prompt = dataset[0]["prompt"]
option_b = dataset[0]["B"]

#Generate embeddings
prompt_embedding = model.encode(prompt, convert_to_tensor=True)
option_b_embedding = model.encode(option_b, convert_to_tensor=True)

# Cosine similarity
similarity = util.cos_sim(prompt_embedding, option_b_embedding)

print(f'similarity score: {round(similarity.item(), 4)}')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


similarity score: 0.7658


### ***10. Build two complete ranking pipelines evaluating every row in train.csv.***
***Pipeline 1: Use the TF-IDF cosine similarity approach from Milestone 1.***

***Pipeline2: Use the sentence-transformers/all-MiniLM-L6-v2 model to generate embeddings for the prompt and all five options. Rank options using cosine similarity to form Top-3 predictions.***

**First, what is the final MAP@3 score of the all-MiniLM-L6-v2 pipeline across the entire training set? Second, count the number of questions for which the correct answer is NOT present in the TF-IDF Top-3 predictions BUT IS present in the MiniLM Top-3 predictions. What is this exact resulting count?**


## **Zero-shot classification concepts**

####  ***11. Initialize the Hugging Face pipeline for "zero-shot-classification" (it will default to facebook/bart-large-mnli). For the prompt of the 2nd row (index 1), pass Options A, B, and C as the candidate_labels. What is the probability score given to the top-ranked option? (Round to 4 decimal places).***

In [37]:
from datasets import load_dataset
from transformers import pipeline



# dataset = ...

classifier = pipeline("zero-shot-classification")

row = dataset[1]

sequence = row["prompt"]
candidate_labels = [row["A"], row["B"], row["C"]]

result = classifier(
    sequence,
    candidate_labels=candidate_labels
)

print(result)
print("Top score:", round(result["scores"][0], 4))

No model was supplied, defaulted to facebook/bart-large-mnli and revision d7645e1.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

{'sequence': 'What is accelerator-based light-ion fusion?', 'labels': ['Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively easy to implement and can be done in an efficient manner, requiring only a vacuum tube, a pair of electrodes, and a high-voltage transformer. Fusion can be observed with as little as 10 kV between the electrodes.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively difficult to implement and requires a complex system of vacuum tubes, electrodes, and transformers. Fusion can be observed with as little as 100 kV between the electrodes.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce heavy-ion

#### ***Run the exact same zero-shot classification as the previous question, but this time pass the argument multi_label=True. What is the absolute difference between the sum of the 3 probabilities in the previous question (which uses Softmax) and the sum of the 3 probabilities in this question (which uses independent Sigmoids)?***

In [38]:
result_softmax = classifier(
    sequence,
    candidate_labels=candidate_labels
)


result_sigmoid = classifier(
    sequence,
    candidate_labels=candidate_labels,
    multi_label=True
)

softmax_sum = sum(result_softmax["scores"])
sigmoid_sum = sum(result_sigmoid["scores"])

difference = abs(softmax_sum - sigmoid_sum)

print("Softmax sum :", softmax_sum)
print("Sigmoid sum :", sigmoid_sum)
print("Difference  :", round(difference, 4))

Softmax sum : 0.9999999701976776
Sigmoid sum : 0.0005096072964079212
Difference  : 0.9995


### ***12. Let's try Generative AI instead of Classification. Load a Small Language Model like google/flan-t5-small using the Hugging Face pipeline("text2text-generation"). Construct the following exact string for row index 0: "Question: [prompt]. Is the correct answer A: [A] or B: [B]? Answer with just the letter A or B." Pass this string to the pipeline, setting max_new_tokens=5. What is the exact string output returned by the model?***

In [43]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM


dataset = load_dataset("csv", data_files="/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")["train"]

# FLAN-T5
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")


row = dataset[0]

text = (
    f"Question: {row['prompt']}. "
    f"Is the correct answer A: {row['A']} or B: {row['B']}? "
    f"Answer with just the letter A or B."
)

# Tokenize
inputs = tokenizer(text, return_tensors="pt")

# Generate
outputs = model.generate(**inputs, max_new_tokens=5)

# Decode

answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(answer)

Generating train split: 0 examples [00:00, ? examples/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

B
